# ICBHI 2017 Respiratory Sound Classification
### Levels: Healthy/Unhealthy (L1) | Sound Anomaly (L2) | K-Means Clustering

In [1]:
# Core imports
import os
import numpy as np
import pandas as pd
import librosa
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, silhouette_score
)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Visualization
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Output directory (Kaggle working dir)
OUT = '/kaggle/working/results'
os.makedirs(OUT, exist_ok=True)
print(f"Results will be saved to: {OUT}")

Results will be saved to: /kaggle/working/results


In [2]:
# Dataset paths
dataset_root = '/kaggle/input/datasets/nimalanparameshwaran/icbhi-2017-challenge-respiratory-sound-database'
csv_path = '/kaggle/input/datasets/santhoshsanka/dataaa/patient_diagnosis.csv'

# Load patient diagnoses; assign binary Level-1 label (0=Healthy, 1=Unhealthy)
diag_df = pd.read_csv(csv_path, header=None, names=['Patient_ID', 'Diagnosis'])
diag_df['Patient_ID'] = diag_df['Patient_ID'].astype(str)
diag_df['Level1'] = diag_df['Diagnosis'].apply(lambda x: 0 if x == 'Healthy' else 1)
diagnosis_dict = dict(zip(diag_df['Patient_ID'], diag_df['Level1']))

TARGET_SR    = 22050
DURATION     = 2.5
TARGET_LEN   = int(TARGET_SR * DURATION)

features, y_lv1, y_lv2 = [], [], []
print("Extracting MFCC features — this may take a few minutes...")

for root, dirs, files in os.walk(dataset_root):
    for file in files:
        if not (file.endswith('.txt') and not file.startswith('.')): continue
        base_name  = file.split('.')[0]
        patient_id = base_name.split('_')[0]
        if patient_id not in diagnosis_dict: continue

        lv1_label = diagnosis_dict[patient_id]
        txt_path  = os.path.join(root, file)
        wav_path  = os.path.join(root, base_name + '.wav')
        if not os.path.exists(wav_path): continue

        # Parse annotation file (tab or space delimited)
        try:
            annotations = pd.read_csv(txt_path, sep='\t', header=None,
                                      names=['Start','End','Crackle','Wheeze'])
        except:
            try:
                annotations = pd.read_csv(txt_path, sep=' ', header=None,
                                          names=['Start','End','Crackle','Wheeze'])
            except: continue

        try:
            y_audio, sr = librosa.load(wav_path, sr=TARGET_SR)
        except: continue

        for _, row in annotations.iterrows():
            try:
                c, w = int(row['Crackle']), int(row['Wheeze'])
                # Level-2: 0=None, 1=Crackle, 2=Wheeze, 3=Both
                lv2_label = c * 1 + w * 2 if not (c and w) else 3
                if c == 1 and w == 0: lv2_label = 1
                elif c == 0 and w == 1: lv2_label = 2
                elif c == 1 and w == 1: lv2_label = 3
                else: lv2_label = 0

                seg = y_audio[int(row['Start']*sr):int(row['End']*sr)]
                if len(seg) == 0: continue
                seg = seg[:TARGET_LEN] if len(seg) > TARGET_LEN else \
                      np.pad(seg, (0, TARGET_LEN - len(seg)))

                # 40 MFCC coefficients → mean + std = 80-dim feature vector
                mfccs = librosa.feature.mfcc(y=seg, sr=sr, n_mfcc=40)
                fv = np.hstack([mfccs.mean(axis=1), mfccs.std(axis=1)])
                features.append(fv); y_lv1.append(lv1_label); y_lv2.append(lv2_label)
            except: continue

X    = np.array(features)
y_lv1 = np.array(y_lv1)
y_lv2 = np.array(y_lv2)
print(f"\nExtraction complete! Cycles: {X.shape[0]}, Features: {X.shape[1]}")

Extracting MFCC features — this may take a few minutes...

Extraction complete! Cycles: 6898, Features: 80


In [3]:
# Scale features; stratified train/test splits for both levels
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_tr1, X_te1, y_tr1, y_te1 = train_test_split(
    X_scaled, y_lv1, test_size=0.2, random_state=42, stratify=y_lv1)
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_scaled, y_lv2, test_size=0.2, random_state=42, stratify=y_lv2)

print(f"L1 train/test: {len(y_tr1)}/{len(y_te1)}")
print(f"L2 train/test: {len(y_tr2)}/{len(y_te2)}")

L1 train/test: 5518/1380
L2 train/test: 5518/1380


In [4]:
def save_confusion_matrix(cm, class_names, model_name, level_tag):
    """Save a styled confusion matrix heatmap to disk."""
    fig, ax = plt.subplots(figsize=(max(5, len(class_names)*1.8),
                                    max(4, len(class_names)*1.6)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f'{model_name} — {level_tag}', fontsize=12)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    fname = f"{OUT}/cm_{level_tag}_{model_name.replace(' ','_').replace('(','').replace(')','')}.png"
    fig.savefig(fname, dpi=120)
    plt.close(fig)


def train_and_evaluate(X_tr, X_te, y_tr, y_te, level_tag, class_names):
    """Train 4 classifiers, save metrics CSV and confusion matrix PNGs."""
    models = {
        'Random_Forest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
        'XGBoost':       xgb.XGBClassifier(eval_metric='mlogloss', random_state=42, n_jobs=-1),
        'LightGBM':      lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
        'SVM_RBF':       SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
    }

    records = []
    print(f"\n{'='*50}\n{level_tag}\n{'='*50}")
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)

        acc = accuracy_score(y_te, y_pred)
        f1  = f1_score(y_te, y_pred, average='weighted')
        print(f"\n{name}  |  Acc: {acc*100:.2f}%  |  F1: {f1*100:.2f}%")
        print(classification_report(y_te, y_pred, target_names=class_names))

        # Confusion matrix PNG
        cm = confusion_matrix(y_te, y_pred)
        save_confusion_matrix(cm, class_names, name, level_tag)

        records.append({'Model': name, 'Level': level_tag,
                        'Accuracy': round(acc*100, 3),
                        'F1_Weighted': round(f1*100, 3)})

    return pd.DataFrame(records)

In [5]:
# Level 1 — Healthy vs Unhealthy
df_l1 = train_and_evaluate(
    X_tr1, X_te1, y_tr1, y_te1,
    'L1_Healthy_Unhealthy',
    ['Healthy', 'Unhealthy']
)


L1_Healthy_Unhealthy

Random_Forest  |  Acc: 96.67%  |  F1: 95.76%
              precision    recall  f1-score   support

     Healthy       1.00      0.28      0.44        64
   Unhealthy       0.97      1.00      0.98      1316

    accuracy                           0.97      1380
   macro avg       0.98      0.64      0.71      1380
weighted avg       0.97      0.97      0.96      1380


XGBoost  |  Acc: 97.68%  |  F1: 97.59%
              precision    recall  f1-score   support

     Healthy       0.80      0.67      0.73        64
   Unhealthy       0.98      0.99      0.99      1316

    accuracy                           0.98      1380
   macro avg       0.89      0.83      0.86      1380
weighted avg       0.98      0.98      0.98      1380


LightGBM  |  Acc: 97.97%  |  F1: 97.85%
              precision    recall  f1-score   support

     Healthy       0.86      0.67      0.75        64
   Unhealthy       0.98      0.99      0.99      1316

    accuracy                     

In [6]:
# Level 2 — Sound anomaly (None / Crackle / Wheeze / Both)
df_l2 = train_and_evaluate(
    X_tr2, X_te2, y_tr2, y_te2,
    'L2_Sound_Anomaly',
    ['None', 'Crackle', 'Wheeze', 'Both']
)


L2_Sound_Anomaly

Random_Forest  |  Acc: 72.39%  |  F1: 69.66%
              precision    recall  f1-score   support

        None       0.70      0.93      0.80       729
     Crackle       0.75      0.64      0.69       373
      Wheeze       0.83      0.31      0.45       177
        Both       0.81      0.25      0.38       101

    accuracy                           0.72      1380
   macro avg       0.77      0.53      0.58      1380
weighted avg       0.74      0.72      0.70      1380


XGBoost  |  Acc: 72.32%  |  F1: 70.66%
              precision    recall  f1-score   support

        None       0.73      0.89      0.81       729
     Crackle       0.70      0.65      0.68       373
      Wheeze       0.72      0.40      0.51       177
        Both       0.67      0.33      0.44       101

    accuracy                           0.72      1380
   macro avg       0.71      0.57      0.61      1380
weighted avg       0.72      0.72      0.71      1380


LightGBM  |  Acc: 72.25% 

In [7]:
# Save all classification metrics to CSV
all_metrics = pd.concat([df_l1, df_l2], ignore_index=True)
metrics_path = f"{OUT}/all_metrics.csv"
all_metrics.to_csv(metrics_path, index=False)
print("Metrics saved →", metrics_path)
display(all_metrics)

Metrics saved → /kaggle/working/results/all_metrics.csv


,Model,Level,Accuracy,F1_Weighted
0,Random_Forest,L1_Healthy_Unhealthy,96.667,95.760
1,XGBoost,L1_Healthy_Unhealthy,97.681,97.587
2,LightGBM,L1_Healthy_Unhealthy,97.971,97.852
3,SVM_RBF,L1_Healthy_Unhealthy,98.768,98.713
4,Random_Forest,L2_Sound_Anomaly,72.391,69.657
5,XGBoost,L2_Sound_Anomaly,72.319,70.656
6,LightGBM,L2_Sound_Anomaly,72.246,70.474
7,SVM_RBF,L2_Sound_Anomaly,74.928,74.375


In [8]:
# Bar chart comparing model accuracies across both levels
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (grp_name, grp) in zip(axes, all_metrics.groupby('Level')):
    bars = ax.bar(grp['Model'], grp['Accuracy'], color=sns.color_palette('Set2', len(grp)))
    ax.set_title(grp_name.replace('_', ' '), fontsize=12)
    ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0, 100)
    ax.set_xticklabels(grp['Model'], rotation=20, ha='right')
    for bar, val in zip(bars, grp['Accuracy']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
plt.suptitle('Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f"{OUT}/accuracy_comparison.png", dpi=150)
plt.show()
print("Saved accuracy_comparison.png")

Saved accuracy_comparison.png


## K-Means Clustering Analysis
Unsupervised clustering on MFCC features for all label levels.
Covers: Elbow curve, Silhouette scores, PCA 2D projection, t-SNE 2D projection, cluster composition heatmaps.

In [9]:
# PCA to 50 dims for faster distance computation (retains >95% variance typically)
pca_50 = PCA(n_components=50, random_state=42)
X_pca50 = pca_50.fit_transform(X_scaled)
print(f"PCA 50-dim explained variance: {pca_50.explained_variance_ratio_.sum()*100:.1f}%")

PCA 50-dim explained variance: 95.9%


In [10]:
# Elbow + Silhouette curve: k = 2..10
K_RANGE   = range(2, 11)
inertias, silhouettes = [], []

print("Running KMeans for k=2..10 ...")
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca50)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_pca50, labels, sample_size=3000, random_state=42))
    print(f"  k={k}  inertia={km.inertia_:.0f}  silhouette={silhouettes[-1]:.4f}")

# Save elbow data
elbow_df = pd.DataFrame({'k': list(K_RANGE), 'Inertia': inertias, 'Silhouette': silhouettes})
elbow_df.to_csv(f"{OUT}/kmeans_elbow_silhouette.csv", index=False)
print("Saved kmeans_elbow_silhouette.csv")

Running KMeans for k=2..10 ...
  k=2  inertia=458884  silhouette=0.1621
  k=3  inertia=416381  silhouette=0.1257
  k=4  inertia=382799  silhouette=0.1298
  k=5  inertia=363286  silhouette=0.1359
  k=6  inertia=349445  silhouette=0.1345
  k=7  inertia=337597  silhouette=0.0967
  k=8  inertia=326534  silhouette=0.1050
  k=9  inertia=318185  silhouette=0.0913
  k=10  inertia=309471  silhouette=0.0892
Saved kmeans_elbow_silhouette.csv


In [11]:
# Plot Elbow + Silhouette
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(list(K_RANGE), inertias, 'bo-', linewidth=2)
ax1.set_xlabel('k'); ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Curve', fontsize=12); ax1.grid(alpha=0.3)

ax2.plot(list(K_RANGE), silhouettes, 'rs-', linewidth=2)
ax2.axvline(np.argmax(silhouettes)+2, color='gray', linestyle='--', alpha=0.7,
            label=f'Best k={np.argmax(silhouettes)+2}')
ax2.set_xlabel('k'); ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score vs k', fontsize=12)
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('KMeans Cluster Quality', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f"{OUT}/kmeans_elbow_silhouette.png", dpi=150)
plt.show()
print("Saved kmeans_elbow_silhouette.png")

Saved kmeans_elbow_silhouette.png


In [12]:
# Choose best k by silhouette; also always run k=2 (matches L1) and k=4 (matches L2)
best_k = int(np.argmax(silhouettes)) + 2
K_FINAL = sorted(set([2, 4, best_k]))
print(f"Best k by silhouette: {best_k}. Running final KMeans for k={K_FINAL}")

km_models   = {}
km_labels_d = {}
for k in K_FINAL:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_labels_d[k] = km.fit_predict(X_pca50)
    km_models[k]   = km
    print(f"  k={k} done")

Best k by silhouette: 2. Running final KMeans for k=[2, 4]
  k=2 done
  k=4 done


In [13]:
# PCA 2D for visualisation (shared projection)
pca_2d = PCA(n_components=2, random_state=42)
X_2d   = pca_2d.fit_transform(X_scaled)

fig, axes = plt.subplots(1, len(K_FINAL), figsize=(6*len(K_FINAL), 5))
if len(K_FINAL) == 1: axes = [axes]

for ax, k in zip(axes, K_FINAL):
    lbl = km_labels_d[k]
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=lbl, cmap='tab10',
                         s=6, alpha=0.5, rasterized=True)
    # Centroids are in PCA-50 space; inverse-transform to 80-dim before projecting
    centers_80 = pca_50.inverse_transform(km_models[k].cluster_centers_)
    centers_2d = pca_2d.transform(centers_80)
    ax.scatter(centers_2d[:, 0], centers_2d[:, 1], c='black',
               marker='X', s=180, zorder=5, label='Centroids')
    ax.set_title(f'KMeans k={k} (PCA 2D)', fontsize=11)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.legend(fontsize=8); ax.grid(alpha=0.2)

plt.suptitle('KMeans Clusters — PCA Projection', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f"{OUT}/kmeans_pca2d.png", dpi=150)
plt.show()
print("Saved kmeans_pca2d.png")

Saved kmeans_pca2d.png


In [14]:
# t-SNE 2D (computationally heavier — sample 3000 points for speed)
TSNE_N = min(3000, X_scaled.shape[0])
rng    = np.random.default_rng(42)
idx    = rng.choice(X_scaled.shape[0], TSNE_N, replace=False)

print(f"Running t-SNE on {TSNE_N} samples...")
tsne   = TSNE(n_components=2, random_state=42, perplexity=40,
              n_iter=1000, learning_rate='auto', init='pca')
X_tsne = tsne.fit_transform(X_scaled[idx])
print("t-SNE done.")

# Plot t-SNE coloured by L1, L2, and best-k cluster labels
label_sets = {
    'L1 (Healthy/Unhealthy)': (y_lv1[idx], ['Healthy','Unhealthy'], 'Set1'),
    'L2 (Sound Anomaly)':     (y_lv2[idx], ['None','Crackle','Wheeze','Both'], 'Set2'),
    f'KMeans k={best_k}':     (km_labels_d[best_k][idx],
                               [f'Cluster {i}' for i in range(best_k)], 'tab10')
}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (title, (lbls, names, cmap)) in zip(axes, label_sets.items()):
    unique = np.unique(lbls)
    palette = sns.color_palette(cmap, len(unique))
    for uid, col in zip(unique, palette):
        mask = lbls == uid
        ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=[col],
                   s=8, alpha=0.6, label=names[uid], rasterized=True)
    ax.set_title(title, fontsize=11)
    ax.legend(markerscale=2, fontsize=8)
    ax.axis('off')

plt.suptitle('t-SNE Projection of MFCC Features', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f"{OUT}/tsne_projections.png", dpi=150)
plt.show()
print("Saved tsne_projections.png")

Running t-SNE on 3000 samples...
t-SNE done.
Saved tsne_projections.png


In [15]:
# Cluster composition heatmaps: how do clusters align with ground-truth labels?
def cluster_composition_heatmap(cluster_labels, gt_labels, gt_names, k, level_tag):
    """Normalised heatmap of cluster vs ground-truth label distribution."""
    df = pd.DataFrame({'Cluster': cluster_labels, 'Label': gt_labels})
    ct = df.groupby(['Cluster','Label']).size().unstack(fill_value=0)
    ct.columns = [gt_names[c] for c in ct.columns]
    ct_norm = ct.div(ct.sum(axis=1), axis=0)   # row-normalise

    fig, ax = plt.subplots(figsize=(max(5, len(gt_names)*1.5), max(3, k*0.8)))
    sns.heatmap(ct_norm, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
                linewidths=0.5, vmin=0, vmax=1)
    ax.set_title(f'Cluster Composition vs {level_tag} (k={k})', fontsize=11)
    ax.set_xlabel('Ground Truth'); ax.set_ylabel('Cluster')
    plt.tight_layout()
    fname = f"{OUT}/cluster_comp_{level_tag}_k{k}.png"
    fig.savefig(fname, dpi=150)
    plt.show()
    print(f"Saved {fname}")

    # Also save raw counts CSV
    ct.to_csv(f"{OUT}/cluster_comp_{level_tag}_k{k}.csv")

# Run for each final k × both label levels
for k in K_FINAL:
    cluster_composition_heatmap(km_labels_d[k], y_lv1, ['Healthy','Unhealthy'], k, 'L1')
    cluster_composition_heatmap(km_labels_d[k], y_lv2, ['None','Crackle','Wheeze','Both'], k, 'L2')

Saved /kaggle/working/results/cluster_comp_L1_k2.png
Saved /kaggle/working/results/cluster_comp_L2_k2.png
Saved /kaggle/working/results/cluster_comp_L1_k4.png
Saved /kaggle/working/results/cluster_comp_L2_k4.png


In [16]:
# Cluster size distribution bar chart for each final k
fig, axes = plt.subplots(1, len(K_FINAL), figsize=(5*len(K_FINAL), 4))
if len(K_FINAL) == 1: axes = [axes]

for ax, k in zip(axes, K_FINAL):
    unique, counts = np.unique(km_labels_d[k], return_counts=True)
    bars = ax.bar([f'C{u}' for u in unique], counts,
                  color=sns.color_palette('tab10', k))
    ax.set_title(f'Cluster Sizes  k={k}', fontsize=11)
    ax.set_xlabel('Cluster'); ax.set_ylabel('Count')
    for b, cnt in zip(bars, counts):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+10,
                str(cnt), ha='center', va='bottom', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('KMeans Cluster Sizes', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f"{OUT}/kmeans_cluster_sizes.png", dpi=150)
plt.show()
print("Saved kmeans_cluster_sizes.png")

Saved kmeans_cluster_sizes.png


In [17]:
# Centroid feature heatmap (mean MFCC values per cluster)
for k in K_FINAL:
    centers_orig = scaler.inverse_transform(
        pca_50.inverse_transform(km_models[k].cluster_centers_)
    )  # shape (k, 80): first 40 = MFCC mean, next 40 = MFCC std

    fig, axes = plt.subplots(1, 2, figsize=(14, max(3, k*0.9)))
    for ax, (start, end, label) in zip(axes,
            [(0,40,'MFCC Mean'), (40,80,'MFCC Std')]):
        data = centers_orig[:, start:end]
        sns.heatmap(data, ax=ax, cmap='coolwarm', center=0,
                    xticklabels=[f'C{i+1}' for i in range(end-start)],
                    yticklabels=[f'Cluster {i}' for i in range(k)])
        ax.set_title(f'Centroid {label}  (k={k})', fontsize=10)
        ax.set_xlabel('MFCC Coefficient Index')

    plt.tight_layout()
    fig.savefig(f"{OUT}/centroid_heatmap_k{k}.png", dpi=150)
    plt.show()
    print(f"Saved centroid_heatmap_k{k}.png")

Saved centroid_heatmap_k2.png
Saved centroid_heatmap_k4.png


In [18]:
# Save cluster assignments CSV (sample-level: kmeans label + ground truth labels)
cluster_df = pd.DataFrame({'L1_Label': y_lv1, 'L2_Label': y_lv2})
for k in K_FINAL:
    cluster_df[f'KMeans_k{k}'] = km_labels_d[k]

cluster_df.to_csv(f"{OUT}/cluster_assignments.csv", index=False)
print("Saved cluster_assignments.csv")
display(cluster_df.head(10))

Saved cluster_assignments.csv


,L1_Label,L2_Label,KMeans_k2,KMeans_k4
0,1,0,1,3
1,1,0,1,1
2,1,0,1,1
3,1,0,1,1
4,1,0,1,1
5,1,0,1,1
6,1,0,1,1
7,1,0,1,1
8,1,3,1,1
9,1,2,1,1


In [19]:
# Summary of all saved outputs
saved = sorted(os.listdir(OUT))
print(f"\n{'='*55}")
print(f"All outputs saved to: {OUT}")
print(f"{'='*55}")
for f in saved:
    fpath = os.path.join(OUT, f)
    size  = os.path.getsize(fpath)
    print(f"  {'📊' if f.endswith('.csv') else '🖼️ '} {f:<55} {size/1024:.1f} KB")


All outputs saved to: /kaggle/working/results
  🖼️  accuracy_comparison.png                                 64.3 KB
  📊 all_metrics.csv                                         0.4 KB
  🖼️  centroid_heatmap_k2.png                                 44.4 KB
  🖼️  centroid_heatmap_k4.png                                 54.1 KB
  📊 cluster_assignments.csv                                 53.9 KB
  📊 cluster_comp_L1_k2.csv                                  0.0 KB
  🖼️  cluster_comp_L1_k2.png                                  26.8 KB
  📊 cluster_comp_L1_k4.csv                                  0.1 KB
  🖼️  cluster_comp_L1_k4.png                                  33.0 KB
  📊 cluster_comp_L2_k2.csv                                  0.1 KB
  🖼️  cluster_comp_L2_k2.png                                  34.8 KB
  📊 cluster_comp_L2_k4.csv                                  0.1 KB
  🖼️  cluster_comp_L2_k4.png                                  44.3 KB
  🖼️  cm_L1_Healthy_Unhealthy_LightGBM.png                  